In [1]:
import pandas as pd

file_path = r"C:\Users\boopa\Desktop\DS\Drone\drone_delivery.csv"

df = pd.read_csv(file_path)

print(df.head())
print(df.info())
print(df.isnull().sum())

    order_id drone_id    drone_type  drone_speed_kmph  payload_weight_kg  \
0  ORD000001   DRN092   Hybrid VTOL              38.9                3.3   
1  ORD000002   DRN081  Single-Rotor              55.7                5.8   
2  ORD000003   DRN061   Multi-Rotor              26.7                9.7   
3  ORD000004   DRN035  Single-Rotor              49.4                6.9   
4  ORD000005   DRN061   Multi-Rotor              26.3                9.3   

   distance_km  battery_efficiency climate_condition  wind_speed_kmph  \
0          4.7                70.1            Cloudy              8.2   
1          2.9                89.1            Cloudy              5.4   
2          4.9                71.0            Cloudy              5.6   
3          1.5                80.6             Windy             17.7   
4          4.2                78.5            Cloudy             18.7   

   temperature_c  humidity_percent   source_area destination_area  \
0           29.4              62.0 

In [2]:
# Separate numeric & categorical columns
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = df.select_dtypes(include=['object']).columns

# Fill numeric missing values with median
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

# Fill categorical missing values with mode
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

C:\Users\boopa\AppData\Local\Temp\ipykernel_22068\395910644.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object']).columns


In [3]:
df = df.drop(columns=['order_id', 'drone_id'])

In [4]:
df = pd.get_dummies(df, drop_first=True)

In [5]:
X = df.drop('ETA_actual_min', axis=1)
y = df['ETA_actual_min']

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Linear Regression

In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

print("Linear Regression MAE:", mean_absolute_error(y_test, y_pred_lr))
print("Linear Regression R2:", r2_score(y_test, y_pred_lr))

Linear Regression MAE: 5.9220957441026
Linear Regression R2: 0.011987806005108959


Random Forest

In [9]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=50,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("Random Forest MAE:", mean_absolute_error(y_test, y_pred_rf))
print("Random Forest R2:", r2_score(y_test, y_pred_rf))

Random Forest MAE: 6.0931849
Random Forest R2: 0.038897605016023906


In [11]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [12]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)  # Output layer for regression
])

C:\Users\boopa\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [13]:
model.compile(
    optimizer='adam',
    loss='mae',   # Mean Absolute Error
    metrics=['mae']
)

In [14]:
history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    verbose=1
)

10000/10000 ━━━━━━━━━━━━━━━━━━━━ 11s 1ms/step - loss: 3.9981 - mae: 3.9981 - val_loss: 4.2135 - val_mae: 4.2135


In [15]:
y_pred_dl = model.predict(X_test_scaled)

from sklearn.metrics import mean_absolute_error, r2_score

print("Deep Learning MAE:", mean_absolute_error(y_test, y_pred_dl))
print("Deep Learning R2:", r2_score(y_test, y_pred_dl))

Deep Learning MAE: 4.391579390959382
Deep Learning R2: -0.006893714156251285
